# Tarea para el Hogar 04

## 5º Corrida - Parte 2:

### A los modelos seleccionados en Parte 1 aplico 5 semillas segundas.

## Total: 60 experimentos

#### A. En la 4º corrida decido dejar de probar hiperparámetros para realizar una evaluación de los modelos obtenidos.

#### B. Hago una selección de 12 modelos según el criterio mencionado en Parte 1.

#### C. Para cada uno de los 12 modelos, realizo una 2º corrida multisemillas, con 5 pares de semillas distintas cada uno, manteniendo el par para que las posteriores comparaciones sean representativas (1º semilla para el modelo, 2º semilla para el undersampling).

#### D. Al finalizar, contaré con 12 modelos a los que habré corrido cada uno con un mismo conjunto de 10 pares de semillas.

#### D. Las semillas, para posterior reproducibilidad, se generan a partir de una función de generación de primos a partir de mi semilla primigenia original.

#### E. Modifico el cuaderno para reemplazar la realización de experimentos mediante iteraciones entre hiperparámetros por la exploración específica de cada modelo seleccionado.


##  1. Overfitting the Public Leaderboard

Leer  https://medium.com/hmif-itb/overfitting-the-leaderboard-da25172ac62e
( 8 minutos )

## 2. Hiperparámetros del LightGBM

Los objetivos de esta tarea son:


*   Aumentar la rentabilidad de la campaña de marketing de retención proactiva de clientes.
*   Generar un mejor modelo optimizando sus hiperparámetros
*   Conceptual : investigar los mas relevantes hiperparámetros de LightGBM
*   Familiarizarse con el uso de máquinas virtuales de Google Colab
*   Ver un pipeline completo de optimización de hiperparámetros y puesta en producción

LightGBM cuenta con mas de 60 hiperparámetros, siendo posible utilizar 40 al mismo tiempo, aunque no razonable.
<br> La documentación oficial de los hiperparámetros de LightGBM es  https://lightgbm.readthedocs.io/en/latest/Parameters.html#core-parameters


Se lo alerta sobre que una Optimizacion sw Hiperparámetros lleva varias horas de corrida, y usted deberá correr VARIAS optimizaciones para descubrir cuales parámetros conviene optimizar.


Es necesario investigar cuales son los hiperparámetros de LightGBM que vale la pena optimizar, ya que los realmente utiles son apenas un reducido subconjunto.
<br>Usted deberá investigar cuales son los hiperparámetros mas relevantes de LightGBM, su primer alternativa es preguntándole a su amigo con capacidades especiales ChatGPT o sus endogámicos familiares Claude, DeepSeek, Gemini, Grok, etc
<br> La segunda alternativa es la propia documentación de LightGBM  https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html


Adicionalmente podra buscar información como la que proveen esta diminuta muestra aleatoria de artículos ligeros:
* https://machinelearningmastery.com/light-gradient-boosted-machine-lightgbm-ensemble/
*  https://medium.com/@sarahzouinina/a-deep-dive-into-lightgbm-how-to-choose-and-tune-parameters-7c584945842e
*  https://www.kaggle.com/code/somang1418/tuning-hyperparameters-under-10-minutes-lgbm
*  https://towardsdatascience.com/beginners-guide-to-the-must-know-lightgbm-hyperparameters-a0005a812702/


<br>  La muestra anterior se brinda a modo de ejemplo, usted deberá buscar muuuuchas  fuentes adicionales de información
<br> Tenga presente que LightGBM es el estado del arte en modelado predictivo para datasets estructurado, que son el 90% del trabajo del 95% de los Data Scientists en Argentina.

El desafío de esta tarea es:
* Qué hiperparparámetros conviene optimizar?  Las recomendaciones de los artículos ligeros es siempre sensata?  Sus autores realmente hicieron experimentos o son siemplemente escritores de entretenimiento carente de base científica?
* Elegidos los hiperparámetros, cual es el  <desde, hasta> que se debe utilizar en la Bayesian Optimization ?
* Realmente vale la pena optimizar 10 o 16 hiperparámetros al mismo tiempo ?  No resulta contraproducente una búsqueda en un espacio de tal alta dimensionalidad ?

#### 2.1  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

### 2.2 Optimizacion Hiperparámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.2.1 Inicio

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Sep 04 11:23:55 AM 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671290,35.9,1473300,78.7,1473300,78.7
Vcells,1242666,9.5,8388608,64.0,1978711,15.1


### 2.2.2 Carga de Librerias

In [3]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: parallel

Loading required package: primes

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘primes’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: primes

Loading required package: rlist

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘rlist’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘XML’


Loading required package: rlist

Loading required package: yaml

Loading required package: lightgbm

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘ligh

### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [4]:
PARAM <- list()
PARAM$experimento <- 5945

PARAM$semilla_primigenia <- 700001

In [5]:
# Generación reproducible de semillas adicionales
#******************************************************************************
# Ahora necesito 5 pares de semillas distintas de las de Parte 1. Genero otras
# 10 semillas pero elimino antes las semillas ya empleadas.
# Asigno los números impares del listado al modelo y los pares al undersampling
#******************************************************************************

if (!require("primes")) install.packages("primes")
require("primes")

primos <- generate_primes(
  min = 100000,
  max = 1000000
)

# Semillas utilizadas en la Parte 1
semillas_usadas <- c(
  445507, 569599,
  172153, 164299,
  665633, 805471,
  629989, 464047,
  343087, 371639
)

# Las elimino del conjunto disponible
primos_disponibles <- setdiff(
  primos,
  semillas_usadas
)

# Genero reproduciblemente otras 10 semillas
set.seed(PARAM$semilla_primigenia)

semillas_nuevas <- sample(
  primos_disponibles,
  10
)

tb_semillas <- data.table(
  par = 6:10,
  semilla_modelo = semillas_nuevas[c(1, 3, 5, 7, 9)],
  semilla_undersampling = semillas_nuevas[c(2, 4, 6, 8, 10)]
)

tb_semillas

par,semilla_modelo,semilla_undersampling
<int>,<int>,<int>
6,445583,569671
7,172169,164309
8,665783,805573
9,630127,464131
10,343153,371719


In [6]:
PARAM$kaggle$competencia <- "utn-2026-inicial"
PARAM$kaggle$cortes <- seq(9000, 12000, by= 500)

In [7]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.5

In [8]:
# Parametros LightGBM

PARAM$hyperparametertuning$xval_folds <- 5

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE, # para reducir warnings
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split= 0, # min_gain_to_split >= 0
  min_sum_hessian_in_leaf= 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1= 0.0, # lambda_l1 >= 0.0
  lambda_l2= 0.0, # lambda_l2 >= 0.0
  max_bin= 31L, # lo debo dejar fijo, no participa de la BO

  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 1.0, # scale_pos_weight > 0.0

  drop_rate= 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop= 50, # <=0 means no limit
  skip_drop= 0.5, # 0.0 <= skip_drop <= 1.0

  extra_trees= FALSE,

  num_iterations= 2000,   #--- Decía 100. Pongo 2000 y luego que corte cuando encuentre no mejora.
  learning_rate= 0.10,  # >=0
  feature_fraction= 1.0, # 0 < ff <= 1.0
  num_leaves= 32, # integer >= 2
  min_data_in_leaf= 20 # integer >= 0
)


### 2.2.4  Preprocesamiento

In [9]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [10]:
# lectura del dataset

dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [11]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [12]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [13]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "azar", "training")
)

In [14]:
nrow(dataset_train)
ncol(dataset_train)

"clase01" %in% colnames(dataset_train)
"azar" %in% colnames(dataset_train)
"training" %in% colnames(dataset_train)

length(campos_buenos)

exists("dtrain")

[1] 164479

[1] 156

[1] TRUE

[1] FALSE

[1] FALSE

[1] 154

[1] FALSE

2.2.5 Configuracion del Grid Search

In [15]:
#*** ESTA REEMPLAZA LA ORIGINAL POR UTILIZACIÓN DE SEMILLAS ***********

Estimar_AUC_lightgbm <- function(
    x,
    dtrain_actual,
    semilla_modelo
) {

  # Los hiperparámetros variables pisan/agregan
  # los valores definidos en param_fijos
  param_completo <- modifyList(
    PARAM$lgbm$param_fijos,
    x
  )

  # La semilla del modelo cambia en cada par
  param_completo$seed <- semilla_modelo

  # También fijo la semilla de R para que la validación cruzada sea comparable
  set.seed(semilla_modelo, kind = "L'Ecuyer-CMRG")

  # Cross validation
  modelocv <- lgb.cv(
    data = dtrain_actual,
    nfold = PARAM$hyperparametertuning$xval_folds,
    stratified = TRUE,
    param = param_completo,
    early_stopping_rounds = 100
  )

  AUC <- modelocv$best_score
  best_iter <- modelocv$best_iter

  message(
    format(Sys.time(), "%a %b %d %X %Y  "),
    "semilla modelo: ", semilla_modelo,
    "  AUC ", AUC,
    "  best_iter ", best_iter
  )

  rm(modelocv)

  gc(
    full = TRUE,
    verbose = FALSE
  )

  return(
    list(
      AUC = AUC,
      best_iter = best_iter
    )
  )
}

In [16]:
tb_modelos <- data.table(
  modelo = sprintf("M%02d", 1:12),

  num_leaves = rep(31L, 12),

  min_data_in_leaf = c(
    500L, 500L, 500L, 500L,
    500L, 500L, 500L, 500L,
    500L, 500L, 300L, 500L
  ),

  learning_rate = c(
    .01, .01, .01, .01,
    .01, .01, .01, .01,
    .01, .01, .01, .02
  ),

  feature_fraction = c(
    .60, .50, .50, .50,
    .50, .50, .60, .60,
    .60, .40, .60, .50
  ),

  bagging_fraction = c(
    .75, .90, .90, .90,
    .90, .90, .90, .90,
    .75, .75, .90, .75
  ),

  bagging_freq = rep(1L, 12),

  lambda_l2 = rep(0, 12),

  max_depth = rep(-1L, 12),

  min_gain_to_split = c(
    .01, .05, .01, .05,
    .05, .05, 0, 0,
    .01, .01, .01, 0
  ),

  lambda_l1 = c(
    0, 0, 0, 0,
    1, 1, 1, 0,
    1, 0, 0, 0
  ),

  min_sum_hessian_in_leaf = c(
    .001, .001, .001, 1,
    .001, 1, .001, 1,
    1, .001, .001, .001
  )
)

tb_modelos

modelo,num_leaves,min_data_in_leaf,learning_rate,feature_fraction,bagging_fraction,bagging_freq,lambda_l2,max_depth,min_gain_to_split,lambda_l1,min_sum_hessian_in_leaf
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
M01,31,500,0.01,0.6,0.75,1,0,-1,0.01,0,0.001
M02,31,500,0.01,0.5,0.90,1,0,-1,0.05,0,0.001
M03,31,500,0.01,0.5,0.90,1,0,-1,0.01,0,0.001
M04,31,500,0.01,0.5,0.90,1,0,-1,0.05,0,1.000
M05,31,500,0.01,0.5,0.90,1,0,-1,0.05,1,0.001
M06,31,500,0.01,0.5,0.90,1,0,-1,0.05,1,1.000
M07,31,500,0.01,0.6,0.90,1,0,-1,0.00,1,0.001
M08,31,500,0.01,0.6,0.90,1,0,-1,0.00,0,1.000
M09,31,500,0.01,0.6,0.75,1,0,-1,0.01,1,1.000


In [17]:
tb_evaluaciones <- merge(
  copy(tb_modelos)[, clave_cartesiana := 1L],
  copy(tb_semillas)[, clave_cartesiana := 1L],
  by = "clave_cartesiana",
  allow.cartesian = TRUE
)

tb_evaluaciones[, clave_cartesiana := NULL]

setorder(
  tb_evaluaciones,
  modelo,
  par
)

tb_evaluaciones[, AUC := NA_real_]
tb_evaluaciones[, best_iter := NA_integer_]

nrow(tb_evaluaciones)

[1] 60

In [18]:
head(tb_evaluaciones, 10)

modelo,num_leaves,min_data_in_leaf,learning_rate,feature_fraction,bagging_fraction,bagging_freq,lambda_l2,max_depth,min_gain_to_split,lambda_l1,min_sum_hessian_in_leaf,par,semilla_modelo,semilla_undersampling,AUC,best_iter
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<dbl>,<int>
M01,31,500,0.01,0.6,0.75,1,0,-1,0.01,0,0.001,6,445583,569671,NA,NA
M01,31,500,0.01,0.6,0.75,1,0,-1,0.01,0,0.001,7,172169,164309,NA,NA
M01,31,500,0.01,0.6,0.75,1,0,-1,0.01,0,0.001,8,665783,805573,NA,NA
M01,31,500,0.01,0.6,0.75,1,0,-1,0.01,0,0.001,9,630127,464131,NA,NA
M01,31,500,0.01,0.6,0.75,1,0,-1,0.01,0,0.001,10,343153,371719,NA,NA
M02,31,500,0.01,0.5,0.90,1,0,-1,0.05,0,0.001,6,445583,569671,NA,NA
M02,31,500,0.01,0.5,0.90,1,0,-1,0.05,0,0.001,7,172169,164309,NA,NA
M02,31,500,0.01,0.5,0.90,1,0,-1,0.05,0,0.001,8,665783,805573,NA,NA
M02,31,500,0.01,0.5,0.90,1,0,-1,0.05,0,0.001,9,630127,464131,NA,NA


In [19]:
#Probamos las tablas creadas. En este experimento, con 12 modelos a probar cada
#uno con 5 semillas, debemos obtener: 12 / 5 / 60 / 60
nrow(tb_modelos)
nrow(tb_semillas)
nrow(tb_evaluaciones)
sum(is.na(tb_evaluaciones$AUC))

[1] 12

[1] 5

[1] 60

[1] 60

In [20]:
# SEGUNDA EVALUACION MULTISEMILLA DE LOS 12 MODELOS
# 12 modelos x 5 nuevos pares de semillas = 60 evaluaciones
# Pares utilizados: 6 a 10
#------------------------------------------------------------

archivo_checkpoint <- "tb_grid_search_05_multisemilla_parte2.txt"

#------------------------------------------------------------
# RECUPERO RESULTADOS SI EXISTE UN CHECKPOINT ANTERIOR
#------------------------------------------------------------

if (file.exists(archivo_checkpoint)) {

  tb_checkpoint <- fread(
    archivo_checkpoint
  )

  if (nrow(tb_checkpoint) == nrow(tb_evaluaciones)) {

    tb_evaluaciones[, AUC := tb_checkpoint$AUC]
    tb_evaluaciones[, best_iter := tb_checkpoint$best_iter]

    message(
      "Checkpoint recuperado: ",
      sum(!is.na(tb_evaluaciones$AUC)),
      " evaluaciones ya realizadas."
    )
  }
}

#------------------------------------------------------------
# RECORRO LOS PARES 6 A 10
#------------------------------------------------------------

for (p in tb_semillas$par) {

  indices_par <- which(
    tb_evaluaciones$par == p
  )

  if (all(!is.na(tb_evaluaciones$AUC[indices_par]))) {

    message(
      "Par ",
      p,
      " ya estaba completo. Se omite."
    )

    next
  }

  #----------------------------------------------------------
  # TOMO LAS DOS SEMILLAS DEL PAR
  #----------------------------------------------------------

  semilla_modelo_actual <- tb_semillas[
    par == p,
    semilla_modelo
  ]

  semilla_undersampling_actual <- tb_semillas[
    par == p,
    semilla_undersampling
  ]

  message("==================================================")

  message(
    "PAR ", p,
    " | semilla modelo = ", semilla_modelo_actual,
    " | semilla undersampling = ", semilla_undersampling_actual
  )

  message("==================================================")


  #----------------------------------------------------------
  # GENERO EL UNDERSAMPLING PARA ESTE PAR DE SEMILLAS
  #----------------------------------------------------------

  set.seed(
    semilla_undersampling_actual,
    kind = "L'Ecuyer-CMRG"
  )

  dataset_train[, azar := runif(nrow(dataset_train))]

  dataset_train[, training := 0L]

  dataset_train[
    foto_mes %in% c(202107) &
      (
        azar <= PARAM$trainingstrategy$undersampling |
          clase_ternaria %in% c("BAJA+1", "BAJA+2")
      ),
    training := 1L
  ]

  #----------------------------------------------------------
  # CREO EL DATASET LIGHTGBM PARA ESTE UNDERSAMPLING
  #----------------------------------------------------------

  dtrain_actual <- lgb.Dataset(
    data = data.matrix(
      dataset_train[
        training == 1L,
        campos_buenos,
        with = FALSE
      ]
    ),
    label = dataset_train[
      training == 1L,
      clase01
    ],
    free_raw_data = FALSE
  )

  #----------------------------------------------------------
  # RECORRO LOS 12 MODELOS DEL PAR
  #----------------------------------------------------------

  for (i in indices_par) {

    # Si esta evaluación ya existe en el checkpoint,
    # no se vuelve a calcular
    if (!is.na(tb_evaluaciones$AUC[i])) {
      next
    }

    modelo_actual <- tb_evaluaciones[i, modelo]

    message(
      "Evaluando ",
      modelo_actual,
      " | par ",
      p
    )

    hiperparametros_actuales <- as.list(
      tb_evaluaciones[
        i,
        .(
          num_leaves,
          min_data_in_leaf,
          learning_rate,
          feature_fraction,
          bagging_fraction,
          bagging_freq,
          lambda_l2,
          max_depth,
          min_gain_to_split,
          lambda_l1,
          min_sum_hessian_in_leaf
        )
      ]
    )

    resultado <- Estimar_AUC_lightgbm(
      hiperparametros_actuales,
      dtrain_actual,
      semilla_modelo_actual
    )

    tb_evaluaciones[
      i,
      `:=`(
        AUC = resultado$AUC,
        best_iter = resultado$best_iter
      )
    ]

    #--------------------------------------------------------
    # GUARDO CHECKPOINT DESPUES DE CADA EVALUACION
    #--------------------------------------------------------

    fwrite(
      tb_evaluaciones,
      archivo_checkpoint,
      sep = "\t"
    )
  }

  # Libero el dataset LightGBM correspondiente a este par
  rm(dtrain_actual)

  gc(
    full = TRUE,
    verbose = FALSE
  )
}



PAR 6 | semilla modelo = 445583 | semilla undersampling = 569671


Evaluando M01 | par 6

Fri Sep 04 11:51:09 AM 2026  semilla modelo: 445583  AUC 0.931878759524659  best_iter 634

Evaluando M02 | par 6

Fri Sep 04 11:54:25 AM 2026  semilla modelo: 445583  AUC 0.931800951300359  best_iter 733

Evaluando M03 | par 6

Fri Sep 04 11:57:51 AM 2026  semilla modelo: 445583  AUC 0.931629903972689  best_iter 733

Evaluando M04 | par 6

Fri Sep 04 12:01:11 PM 2026  semilla modelo: 445583  AUC 0.931751909343591  best_iter 731

Evaluando M05 | par 6

Fri Sep 04 12:04:51 PM 2026  semilla modelo: 445583  AUC 0.932158343175171  best_iter 846

Evaluando M06 | par 6

Fri Sep 04 12:08:13 PM 2026  semilla modelo: 445583  AUC 0.932124985724889  best_iter 750

Evaluando M07 | par 6

Fri Sep 04 12:10:57 PM 2026  semilla modelo: 445583  AUC 0.931573157639778  best_iter 756

Evaluando M08 | par 6

Fri Sep 04 12:13:36 PM 2026  semilla modelo: 445583  AUC 0.931521263168218  best_iter 756

Evaluando M09 | par 

In [21]:
# veo que tiene la tabla DESPUES de procesar
tb_evaluaciones

In [22]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
#********* NO EJECUTAR **********************************
#*** Primero analizar resultados entre semillas *********

## 2.3  Produccion

### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [23]:
setwd("/content/buckets/b1/exp")
experimento <- paste0("exp", PARAM$experimento)
dir.create(experimento, showWarnings= FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

#### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización de hiperparametros

In [24]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [25]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [26]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

#### Final Training Hyperparameters

In [27]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

ERROR: Error in modifyList(PARAM$lgbm$param_fijos, PARAM$out$lgbm$mejores_hiperparametros): is.list(val) is not TRUE


#### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)

In [ ]:
  # entreno LightGBM

  modelo_final <- lgb.train(
    data= dtrain,
    param= param_normalizado
  )

In [ ]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(modelo_final))
archivo_importancia <- "impo.txt"

fwrite(tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

In [ ]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(modelo_final, "modelo.txt" )

### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes == 202109]

# aplico el modelo a los datos nuevos
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

#### Tabla Prediccion

In [ ]:
# tabla de prediccion

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion ]

# grabo las probabilidad del modelo
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

Kaggle Competition Submit

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
  Sys.sleep(45)
}

In [ ]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

Finalmente usted deberá cargar el resultado de su corrida en la Google Sheet Colaborativa,  hoja **TareaHogar-04**
<br> Siéntase libre de agregar las columnas que hagan falta a la planilla

Seguramente usted realice varias corridas de este script con distintos conjuntos de hiperparámetros, siempre cambiandole el nombre al script  y también cambiando el nombre del experimento,  deberá TODAS esas corridas en distintas lineas de la  Google Sheet Colaborativa, hoja **TareaHogar-04**

Siéntase libre de agregar columnas a la hoja **TareaHogar-04**  en caso de ser necesario.